In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS retailpulse.silver;

In [0]:
%sql
SHOW SCHEMAS IN retailpulse;

In [0]:
%sql
DESCRIBE retailpulse.bronze.customers;

In [0]:

from pyspark.sql.types import *
from pyspark.sql import functions as F


# ============================================================
# 1. CONFIGURATION
# ============================================================

source_table = "retailpulse.bronze.customers"
target_table = "retailpulse.silver.customers"
rejected_table = "retailpulse.silver.customers_rejected"


# ============================================================
# 2. DEFINE CUSTOMER SCHEMA
# ============================================================

customer_schema = StructType([
    StructField("customer_id", StringType(), False),
    StructField("customer_unique_id", StringType(), False),
    StructField("customer_zip_code_prefix", StringType(), True),
    StructField("customer_city", StringType(), True),
    StructField("customer_state", StringType(), False),
    StructField("source_file", StringType(), True),
    StructField("ingested_at", TimestampType(), True),
    StructField("run_id", StringType(), True)
])


# ============================================================
# 3. READ BRONZE DATA
# ============================================================

customers_df = spark.read.table(source_table)

customers_df.limit(10).display()
customers_df.printSchema()

total_records_count = customers_df.count()

print("Total Bronze records:", total_records_count)


# ============================================================
# 4. STRING STANDARDIZATION
# ============================================================

string_columns = [
    field.name
    for field in customers_df.schema.fields
    if isinstance(field.dataType, StringType)
]

# Remove leading and trailing spaces
for c in string_columns:
    customers_df = customers_df.withColumn(
        c, F.trim(F.col(c))
    )

# Standardize city and state
customers_df = customers_df.withColumns({
    "customer_city": F.initcap(F.col("customer_city")),
    "customer_state": F.upper(F.col("customer_state"))
})

customers_df.limit(10).display()


# ============================================================
# 5. DATA QUALITY PROFILING
# ============================================================

# NULL count for every column
null_counts = [
    F.coalesce(
        F.sum(
            F.when(F.col(c).isNull(), 1).otherwise(0)
        ),
        F.lit(0)
    ).alias(c)
    for c in customers_df.columns
]

null_counts_dict = (
    customers_df.agg(*null_counts).first().asDict()
)


# Empty-string count for string columns
emptystring_counts = [
    F.coalesce(
        F.sum(
            F.when(F.col(c) == "", 1).otherwise(0)
        ),
        F.lit(0)
    ).alias(c)
    for c in string_columns
]

emptystring_counts_dict = (
    customers_df.agg(*emptystring_counts).first().asDict()
)

print("NULL COUNTS:", null_counts_dict)
print("EMPTY STRING COUNTS:", emptystring_counts_dict)


# ============================================================
# 6. MANDATORY FIELD VALIDATION
# ============================================================

mandatory_columns = [
    "customer_id",
    "customer_unique_id",
    "customer_state"
]

invalid_records_df = customers_df.filter(
    (F.col("customer_id").isNull()) |
    (F.col("customer_id") == "") |

    (F.col("customer_unique_id").isNull()) |
    (F.col("customer_unique_id") == "") |

    (F.col("customer_state").isNull()) |
    (F.col("customer_state") == "")
)

valid_records_df = customers_df.filter(
    (F.col("customer_id").isNotNull()) &
    (F.col("customer_id") != "") &

    (F.col("customer_unique_id").isNotNull()) &
    (F.col("customer_unique_id") != "") &

    (F.col("customer_state").isNotNull()) &
    (F.col("customer_state") != "")
)


# ============================================================
# 8. RECORD COUNTS
# ============================================================

invalid_records_count = invalid_records_df.count()
valid_records_count = valid_records_df.count()

print("Bronze records:", total_records_count)
print("Valid records:", valid_records_count)
print("Invalid records:", invalid_records_count)


# ============================================================
# 9. DATA RECONCILIATION
# ============================================================

reconciled_count = (
    valid_records_count + invalid_records_count
)

print("Reconciled records:", reconciled_count)

if total_records_count == reconciled_count:
    print("RECONCILIATION PASSED")
else:
    print("RECONCILIATION FAILED")
